## Эксперименты с реранкером

In [ ]:
import json
import numpy as np
import pandas as pd
from datetime import datetime
from typing import List, Dict, Any, Optional
from collections import defaultdict
from sentence_transformers import CrossEncoder
import torch

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Используется устройство: {device}")

reranker = CrossEncoder(
    "Qwen/Qwen3-Reranker-0.6B",
    device=device,
    trust_remote_code=True
)

In [ ]:
def rerank_points(
    query: str,
    points: List[Dict],
    top_k: int = 5,
    batch_size: int = 32
) -> List[Dict]:
    """
    Реранкинг точек с использованием CrossEncoder на GPU
    """
    if not points:
        return []
    
    pairs = [(query, point['text']) for point in points]
    
    scores = []
    for i in range(0, len(pairs), batch_size):
        batch_pairs = pairs[i:i+batch_size]
        batch_scores = reranker.predict(batch_pairs, show_progress_bar=False)
        if isinstance(batch_scores, np.ndarray):
            scores.extend(batch_scores.tolist())
        else:
            scores.extend(batch_scores)

    for point, score in zip(points, scores):
        point['rerank_score'] = float(score)
    
    # Сортируем по новому score
    points.sort(key=lambda x: x['rerank_score'], reverse=True)
    
    return points[:top_k]

In [ ]:
def evaluate_with_reranker_from_file(
    search_results_file: str,
    original_questions_file: str,
    rerank_top_k: int = 5,
    batch_size: int = 32,
    output_file: Optional[str] = None
) -> Dict[str, Any]:
    """
    Оценка качества поиска с реранкингом на основе saved search results (GPU оптимизация)
    
    Args:
        search_results_file: JSON файл с результатами поиска
        original_questions_file: JSON файл с вопросами и релевантными ID
        rerank_top_k: сколько точек оставить после реранкера
        batch_size: размер батча для реранкера
        output_file: файл для сохранения результатов
    """
    print("Загрузка данных...")
    with open(search_results_file, 'r', encoding='utf-8') as f:
        search_results = json.load(f)
    
    with open(original_questions_file, 'r', encoding='utf-8') as f:
        original_data = json.load(f)
    
    relevant_map = {}
    for item in original_data:
        question = item.get('question', '')
        relevant_ids = set(item.get('id', []))
        if question:
            relevant_map[question] = relevant_ids
    
    if output_file is None:
        output_file = f"reranker_eval_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
    
    metrics = {
        'total_queries': len(search_results),
        'relevant_queries': 0,
        'precision': [],
        'recall': [],
        'f1_score': [],
        'success_rate': [],
        'relevant_found_count': [],
        'detailed_results': [],
        'relevance_distribution': defaultdict(int),
        'inference_times': [],  # для замера скорости
    }
    
    for idx, item in enumerate(search_results):
        question = item.get('question', '')
        points = item.get('points', [])
        
        if not question or not points:
            continue
        
        relevant_ids = relevant_map.get(question, set())
        
        if not relevant_ids:
            print(f" {idx+1}/{len(search_results)}: нет релевантных ID для: {question[:50]}...")
            continue
        
        retrieved_ids_before = [p.get('doc_id', p.get('document_id', '')) for p in points[:rerank_top_k]]
        
        import time
        start_time = time.time()
        reranked_points = rerank_points(question, points, rerank_top_k, batch_size)
        inference_time = time.time() - start_time
        metrics['inference_times'].append(inference_time)
        
        retrieved_ids_after = [p.get('doc_id', p.get('document_id', '')) for p in reranked_points]
        
        num_relevant_total = len(relevant_ids)

        relevant_before = set(retrieved_ids_before) & relevant_ids
        num_relevant_before = len(relevant_before)
        num_retrieved_before = len(retrieved_ids_before)
        
        precision_before = num_relevant_before / num_retrieved_before if num_retrieved_before > 0 else 0
        recall_before = num_relevant_before / num_relevant_total if num_relevant_total > 0 else 0
        f1_before = 2 * precision_before * recall_before / (precision_before + recall_before) if (precision_before + recall_before) > 0 else 0
        
        relevant_after = set(retrieved_ids_after) & relevant_ids
        num_relevant_after = len(relevant_after)
        num_retrieved_after = len(retrieved_ids_after)
        
        precision_after = num_relevant_after / num_retrieved_after if num_retrieved_after > 0 else 0
        recall_after = num_relevant_after / num_relevant_total if num_relevant_total > 0 else 0
        f1_after = 2 * precision_after * recall_after / (precision_after + recall_after) if (precision_after + recall_after) > 0 else 0
        success_after = 1 if num_relevant_after > 0 else 0
        
        metrics['precision'].append(precision_after)
        metrics['recall'].append(recall_after)
        metrics['f1_score'].append(f1_after)
        metrics['success_rate'].append(success_after)
        metrics['relevant_found_count'].append(num_relevant_after)
        metrics['relevance_distribution'][num_relevant_after] += 1
        
        if num_relevant_after > 0:
            metrics['relevant_queries'] += 1
        
        detailed_result = {
            'query': question[:100],
            'query_id': idx,
            'relevant_total': num_relevant_total,
            'retrieved_before': num_retrieved_before,
            'relevant_before': num_relevant_before,
            'precision_before': round(precision_before, 4),
            'recall_before': round(recall_before, 4),
            'f1_before': round(f1_before, 4),
            'retrieved_after': num_retrieved_after,
            'relevant_after': num_relevant_after,
            'precision_after': round(precision_after, 4),
            'recall_after': round(recall_after, 4),
            'f1_after': round(f1_after, 4),
            'success': success_after,
            'improvement': round((f1_after - f1_before) / f1_before * 100 if f1_before > 0 else 0, 1),
            'inference_time_ms': round(inference_time * 1000, 2)
        }
        metrics['detailed_results'].append(detailed_result)
        
        print(f"\n{idx+1}/{len(search_results)}: {question[:50]}...")
        print(f"   До реранкера: P={precision_before:.3f}, R={recall_before:.3f}, F1={f1_before:.3f}")
        print(f"   После реранкера: P={precision_after:.3f}, R={recall_after:.3f}, F1={f1_after:.3f}")
        print(f"   Время инференса: {inference_time*1000:.1f}ms")
    
    aggregated = {
        'total_queries': metrics['total_queries'],
        'relevant_queries': metrics['relevant_queries'],
        'success_rate': np.mean(metrics['success_rate']) * 100,
        'avg_precision': np.mean(metrics['precision']),
        'avg_recall': np.mean(metrics['recall']),
        'avg_f1': np.mean(metrics['f1_score']),
        'median_precision': np.median(metrics['precision']),
        'median_recall': np.median(metrics['recall']),
        'median_f1': np.median(metrics['f1_score']),
        'avg_relevant_found': np.mean(metrics['relevant_found_count']),
        'avg_inference_time_ms': np.mean(metrics['inference_times']) * 1000 if metrics['inference_times'] else 0,
        'device': device
    }
    
    print("\n" + "=" * 70)
    print(f"📊 РЕЗУЛЬТАТЫ С РЕРАНКЕРОМ (GPU: {device})")
    print("=" * 70)
    print(f"\n📈 F1-SCORE: {aggregated['avg_f1']:.4f}")
    print(f"📈 ТОЧНОСТЬ (PRECISION): {aggregated['avg_precision']:.4f}")
    print(f"📈 ПОЛНОТА (RECALL): {aggregated['avg_recall']:.4f}")
    print(f"⏱️ Среднее время инференса: {aggregated['avg_inference_time_ms']:.1f}ms на запрос")
    
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write("=" * 80 + "\n")
        f.write(f"ОЦЕНКА РЕРАНКЕРА\n")
        f.write(f"Дата: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Устройство: {device}\n")
        f.write("=" * 80 + "\n\n")
        
        f.write(f"Всего запросов: {aggregated['total_queries']}\n")
        f.write(f"Успешных: {aggregated['relevant_queries']} ({aggregated['success_rate']:.1f}%)\n\n")
        
        f.write("ПОСЛЕ РЕРАНКЕРА:\n")
        f.write(f"  Precision: {aggregated['avg_precision']:.4f}\n")
        f.write(f"  Recall: {aggregated['avg_recall']:.4f}\n")
        f.write(f"  F1: {aggregated['avg_f1']:.4f}\n")
        f.write(f"  Среднее время: {aggregated['avg_inference_time_ms']:.1f}ms\n")

    detailed_df = pd.DataFrame(metrics['detailed_results'])
    detailed_csv = output_file.replace('.txt', '_detailed.csv')
    detailed_df.to_csv(detailed_csv, index=False, encoding='utf-8')
    
    print(f"\n Результаты сохранены в {output_file}")
    print(f" Детальные результаты в {detailed_csv}")
    
    return aggregated

In [ ]:
if __name__ == "__main__":
    results = evaluate_with_reranker_from_file(
        search_results_file="/kaggle/input/datasets/kirillkrd/crunch/search_results_20260427_195011.json",
        original_questions_file="/kaggle/input/datasets/kirillkrd/crunch/questions_with_embeds.json",
        rerank_top_k=5,
        batch_size=1  
    )